# 1. Model behavior modes (train/eval)
### `1. model.train()`
#### What it does: 
* Sets the model to training mode
* Affects layers that behave differently during training

#### Affected layers
| Layer       | Behavior                                 |
| ----------- | ---------------------------------------- |
| `Dropout`   | Randomly drops units                     |
| `BatchNorm` | Uses batch stats & updates running stats |
| `LayerNorm` | Same behavior (no mode difference)       |

#### When to use
* During training
* Before forward + backward

### `2. model.eval()`
#### What it does
* Sets the model to evaluation mode
* Freezes stochastic behavior

#### Affected layers
| Layer       | Behavior              |
| ----------- | --------------------- |
| `Dropout`   | Disabled              |
| `BatchNorm` | Uses running mean/var |

#### Important
* Does NOT disable gradients
* Does NOT stop graph creation

# summary table
| Mode               | Affects layers | Affects autograd | Speed   |
| ------------------ | -------------- | ---------------- | ------- |
| `train()`          | ✅              | ❌                | Normal  |
| `eval()`           | ✅              | ❌                | Normal  |
| Normal forward     | ❌              | ✅                | Slow    |
| `no_grad()`        | ❌              | ✅                | Fast    |
| `inference_mode()` | ❌              | ✅                | Fastest |

# Common misconceptions
| Myth                          | Reality                      |
| ----------------------------- | ---------------------------- |
| `eval()` disables gradients   | ❌ No                         |
| `no_grad()` changes Dropout   | ❌ No                         |
| `inference_mode()` = `eval()` | ❌ Different responsibilities |

### Final takeaway (memorize this)
* `train() / eval()` → how the model behaves
* `grad/inference modes`→ how PyTorch executes

# 2. Autograd / execution modes (context managers)

### `1.Normal mode (default)`
#### example `y = model(x)`
#### Behaviour 
* Gradients tracked
* Computation graph built
* Backprop allowed

#### use 
* training

### `2.torch.no_grad()`

#### example `with torch.no_grad():`
####              `y = model(x)`
#### Behaviour 
* Disables gradient tracking
* Keeps version counters
* Allows temporary grad disable

#### use 
* Validation
* Quick evaluation

### `2.torch.inference_mode() (strongest)`

#### example `with torch.inference_mode():`
####              `y = model(x)`
#### Behaviour 
* Disables autograd
* Disables version counters
* Faster & memory efficient

#### use 
* Deployment
* Production inference
* Final evaluation

# 3. Parameter gradient modes
#### These control whether parameters accumulate gradients.

### `1.param.requires_grad = True (default)`
#### Behaviour 
* Parameter participates in training

#### use 
* training

### `2.param.requires_grad = False`
#### Behaviour 
* Parameter participates in training
#### use 
* Transfer learning
* Freezing layers

TRAINING
│
├── model.train()
│   └── Gradients ON
│
EVALUATION
│
├── model.eval()
│   ├── torch.no_grad()
│   └── torch.inference_mode()
│
DEPLOYMENT
│
├── model.eval()
│   └── torch.inference_mode()  ← mandatory


# ADDITIONAL EXPLANATION OF MODES

## Diff b/w Normal mode and Inference mode
|Normal mode|Inference mode|
|-----------|-------------|
|Pytorch learning from data, tracking gradients & building graphs.|Pytorch only doing inference, NOT tracking gradients & NOT build graphs.|
|Calls model0.forward(x_test)|Avoids storing intermediate tensors|
|Tracks the computation graph (by default)|Prevents version counter updates|
|Enables autograd so gradients can be computed later|Disables autograd|
|Autograd behavior: a: PyTorch records every operation, |Optimizes memory & speed|
|b : Builds a dynamic computation graph,||
|c : Stores intermediate tensors for backprop||
|loss is possible to calculate here |loss calculations is not possible here|
|uses/ Advantages| used/ Advantages|
|1.training|prediction|
|2. When you need .backward()||
|3. When computing gradients manually||
|Disdvantages|Disdvantages|
|1. Extra memory usage||
|2. Slower inference||
|3. Risk of accidentally calling .backward()||
|4. Dropout / BatchNorm may behave incorrectly if not in eval mode||



| Aspect            | Normal Call | `inference_mode()` |
| ----------------- | ----------- | ------------------ |
| Gradient tracking | ✅ Yes       | ❌ No               |
| Computation graph | Built       | Not built          |
| Memory usage      | Higher      | Lower              |
| Speed             | Slower      | Faster             |
| Backprop allowed  | ✅ Yes       | ❌ No               |
| Deployment safe   | ❌ Risky     | ✅ Ideal            |


# torch.eval is different from inference mode
# inference mode is stronger than torch.no_grad()

## model.eval()
What it does:
1. Changes layer behavior
2. Dropout → OFF
3. BatchNorm → uses running stats
   
What it does NOT do:
1. Does NOT disable gradients
2. Does NOT stop graph creation

| Feature         | no_grad    | inference_mode       |
| --------------- | ---------- | -------------------- |
| Disables grad   | ✅          | ✅                    |
| Graph building  | ❌          | ❌                    |
| Version counter | ✅          | ❌                    |
| Speed           | Good       | Best                 |
| Use case        | Validation | Deployment / Testing |

### Mental model (important)
#### Training → “Learn & remember” → model(x)
#### Evaluation → “Observe carefully” → eval() + inference_mode()
#### Deployment → “Just predict fast” → eval() + inference_mode()

#### Example: Driving analogy 🚗

Evaluation:
Test drive the car on a track
→ Helmet on, safety mode ON

Deployment:
Taxi service running 24/7
→ Speed limits, fuel efficiency, zero crashes

Same seatbelt, but stakes are different.

# 4. saving and loading Models

## `1: PRODUCTION MODE: THERE IS A DIFFERENT WAY TO SAVE MODEL FOR  PRODUCTION/INFERENCE` (saving/loading only state_dict).
## `1: DEVELOPER MODE: SAVE/LOAD FULL MODEL FOR  DEVELOPERS` (saving all components like, state_dict, ).


## DETAILS
### `1. torch.save()`
#### format: 
* pickle

### `2. torch.load()`
#### use: 
* load saved PyTorch model

### `3. torch.nn.load_state_dict()`
#### use: 
* load saved state dictionary of the model

